# Transfer Learning para Visión por Computador con PyTorch
Este notebook muestra una versión en PyTorch del flujo de `transfer learning` que ya hemos trabajado con Keras. La idea es reutilizar un modelo preentrenado para clasificar imágenes del dataset Oxford Flowers 102.

## ¿Qué cambia respecto a Keras?
El objetivo es el mismo, pero el estilo de trabajo cambia:

- en vez de `ImageDataGenerator`, usaremos `Dataset` y `DataLoader`
- en vez de `model.fit(...)`, escribiremos el bucle de entrenamiento
- en vez de `tf.keras.applications`, usaremos `torchvision.models`

Conceptualmente seguimos haciendo lo mismo: cargar datos, reutilizar una red preentrenada, adaptar la última capa y entrenar.

In [ ]:
# Librerías
from pathlib import Path
import json

import matplotlib.pyplot as plt
import numpy as np
from PIL import Image
from scipy.io import loadmat

import torch
from torch import nn
from torch.utils.data import Dataset, DataLoader
from torchvision import models, transforms

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("Dispositivo:", device)

## Carga del dataset Oxford Flowers 102
Usaremos las imágenes originales en `jpg` y las etiquetas y particiones oficiales en los ficheros `.mat`.

In [ ]:
data_dir = Path("vision/data/oxford_flowers102")
jpg_dir = data_dir / "jpg"

labels = loadmat(data_dir / "imagelabels.mat")["labels"].squeeze().astype("int64") - 1
setid = loadmat(data_dir / "setid.mat")

train_ids = setid["trnid"].squeeze()
val_ids = setid["valid"].squeeze()
test_ids = setid["tstid"].squeeze()

num_classes = int(labels.max() + 1)
print(f"Clases: {num_classes}")
print(f"Train: {len(train_ids)} | Val: {len(val_ids)} | Test: {len(test_ids)}")

## Transformaciones de imagen
Aquí aplicamos redimensionado y normalización compatibles con modelos preentrenados de `torchvision`.

In [ ]:
img_size = 160
batch_size = 32

train_transform = transforms.Compose([
    transforms.Resize((img_size, img_size)),
    transforms.RandomHorizontalFlip(),
    transforms.ToTensor(),
    transforms.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225])
])

eval_transform = transforms.Compose([
    transforms.Resize((img_size, img_size)),
    transforms.ToTensor(),
    transforms.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225])
])

## Dataset personalizado

In [ ]:
class OxfordFlowersDataset(Dataset):
    def __init__(self, ids, labels, image_dir, transform=None):
        self.ids = ids
        self.labels = labels
        self.image_dir = image_dir
        self.transform = transform

    def __len__(self):
        return len(self.ids)

    def __getitem__(self, idx):
        image_id = int(self.ids[idx])
        label = int(self.labels[image_id - 1])
        image_path = self.image_dir / f"image_{image_id:05d}.jpg"

        image = Image.open(image_path).convert("RGB")
        if self.transform:
            image = self.transform(image)

        return image, label

In [ ]:
train_dataset = OxfordFlowersDataset(train_ids, labels, jpg_dir, transform=train_transform)
val_dataset = OxfordFlowersDataset(val_ids, labels, jpg_dir, transform=eval_transform)
test_dataset = OxfordFlowersDataset(test_ids, labels, jpg_dir, transform=eval_transform)

train_loader = DataLoader(train_dataset, batch_size=batch_size, shuffle=True)
val_loader = DataLoader(val_dataset, batch_size=batch_size, shuffle=False)
test_loader = DataLoader(test_dataset, batch_size=batch_size, shuffle=False)

print("Train batches:", len(train_loader))
print("Val batches:", len(val_loader))
print("Test batches:", len(test_loader))

In [ ]:
# Vista rápida de algunas imágenes del loader
images, targets = next(iter(train_loader))

mean = np.array([0.485, 0.456, 0.406])
std = np.array([0.229, 0.224, 0.225])

plt.figure(figsize=(10, 4))
for i in range(6):
    img = images[i].permute(1, 2, 0).numpy()
    img = np.clip(img * std + mean, 0, 1)

    plt.subplot(2, 3, i + 1)
    plt.imshow(img)
    plt.title(f"clase {int(targets[i])}")
    plt.axis("off")

plt.tight_layout()
plt.show()

## Modelo preentrenado
Usamos `MobileNetV2` preentrenada y sustituimos la última capa para adaptarla a las 102 clases del problema.

In [ ]:
model = models.mobilenet_v2(weights=models.MobileNet_V2_Weights.IMAGENET1K_V1)

# Congelar la base convolucional
for param in model.features.parameters():
    param.requires_grad = False

# Sustituir la última capa clasificadora
in_features = model.classifier[1].in_features
model.classifier[1] = nn.Linear(in_features, num_classes)

model = model.to(device)
model

In [ ]:
criterion = nn.CrossEntropyLoss()
optimizer = torch.optim.Adam(model.classifier[1].parameters(), lr=1e-3)

## Bucle de entrenamiento
A diferencia de Keras, aquí escribimos explícitamente las fases de entrenamiento y validación.

In [ ]:
def run_epoch(model, dataloader, criterion, optimizer=None):
    train = optimizer is not None
    if train:
        model.train()
    else:
        model.eval()

    total_loss = 0.0
    total_correct = 0
    total_samples = 0

    for images, targets in dataloader:
        images = images.to(device)
        targets = targets.to(device)

        if train:
            optimizer.zero_grad()

        with torch.set_grad_enabled(train):
            outputs = model(images)
            loss = criterion(outputs, targets)

            if train:
                loss.backward()
                optimizer.step()

        preds = outputs.argmax(dim=1)
        total_loss += loss.item() * images.size(0)
        total_correct += (preds == targets).sum().item()
        total_samples += images.size(0)

    return total_loss / total_samples, total_correct / total_samples

In [ ]:
epochs = 5
history = {"train_loss": [], "train_acc": [], "val_loss": [], "val_acc": []}

for epoch in range(epochs):
    train_loss, train_acc = run_epoch(model, train_loader, criterion, optimizer)
    val_loss, val_acc = run_epoch(model, val_loader, criterion)

    history["train_loss"].append(train_loss)
    history["train_acc"].append(train_acc)
    history["val_loss"].append(val_loss)
    history["val_acc"].append(val_acc)

    print(
        f"Epoch {epoch+1}/{epochs} | "
        f"train_loss={train_loss:.4f} train_acc={train_acc:.4f} | "
        f"val_loss={val_loss:.4f} val_acc={val_acc:.4f}"
    )

## Evaluación en test

In [ ]:
test_loss, test_acc = run_epoch(model, test_loader, criterion)
print(f"Test loss: {test_loss:.4f}")
print(f"Test acc:  {test_acc:.4f}")

## Curvas de entrenamiento

In [ ]:
plt.figure(figsize=(10, 4))

plt.subplot(1, 2, 1)
plt.plot(history["train_acc"], label="train")
plt.plot(history["val_acc"], label="val")
plt.title("Precisión")
plt.xlabel("Épocas")
plt.legend()

plt.subplot(1, 2, 2)
plt.plot(history["train_loss"], label="train")
plt.plot(history["val_loss"], label="val")
plt.title("Pérdida")
plt.xlabel("Épocas")
plt.legend()

plt.tight_layout()
plt.show()

## Conclusión
La idea de `transfer learning` es exactamente la misma que en Keras: reutilizar una red entrenada previamente y adaptarla a un nuevo problema. Lo que cambia en PyTorch es que tenemos más control explícito sobre los datos, el modelo y el entrenamiento.

In [ ]:
Markdown('''
**Actividad:**

1. Sustituye `MobileNetV2` por `ResNet18` o `EfficientNet` de `torchvision`.
2. Descongela algunas capas y prueba fine-tuning.
3. Compara este flujo con el notebook equivalente en Keras.
4. Intenta identificar qué partes son conceptualmente iguales y cuáles cambian de estilo.
''')